# Week 7: HealthConnect Clinic — Model Testing, Error Analysis & Refinement

## Data Science Track

This notebook builds on the Week 6 candidate model (refined Logistic 
Regression with the `booking_lead_category_v2` feature). The Week 5/6 
pipeline is reproduced below to establish a self-contained starting 
point, before moving into new Week 7 work: systematic testing, error 
analysis, and evidence-based refinement.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\HP\Downloads\HealthConnect_Appointment_Data.csv")

df_model = df[df['appointment_outcome'] != 'Cancelled'].copy()
df_model['target_no_show'] = (df_model['appointment_outcome'] == 'No-Show').astype(int)

df_model = df_model.drop(columns=[
    'appointment_id', 'booking_date', 'appointment_date',
    'waiting_time_minutes', 'appointment_outcome'
])

df_model['reminder_channel'] = df_model['reminder_channel'].fillna('None')
median_distance = df_model['distance_to_clinic_km'].median()
df_model['distance_to_clinic_km'] = df_model['distance_to_clinic_km'].fillna(median_distance)

print("df_model shape:", df_model.shape)

df_model shape: (4737, 14)


In [2]:
df_model['prior_no_show_rate'] = np.where(
    df_model['previous_appointments'] > 0,
    df_model['previous_no_shows'] / df_model['previous_appointments'],
    0
)
df_model['has_prior_history'] = (df_model['previous_appointments'] > 0).astype(int)

def categorize_lead_time_v2(days):
    if days <= 7:
        return 'A_0-7'
    elif days <= 14:
        return 'B_8-14'
    elif days <= 30:
        return 'C_15-30'
    else:
        return 'D_31plus'

df_model['booking_lead_category_v2'] = df_model['booking_lead_days'].apply(categorize_lead_time_v2)

print("Feature engineering complete. Shape:", df_model.shape)

Feature engineering complete. Shape: (4737, 17)


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score)

categorical_cols_v2 = ['gender', 'age_group', 'appointment_type', 'appointment_day',
                        'appointment_time', 'reminder_sent', 'reminder_channel',
                        'booking_lead_category_v2']

df_encoded = pd.get_dummies(df_model, columns=categorical_cols_v2, drop_first=True)

X = df_encoded.drop(columns=['patient_id', 'target_no_show'])
y = df_encoded['target_no_show']
groups = df_encoded['patient_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

numerical_cols = ['age', 'booking_lead_days', 'previous_appointments',
                   'previous_no_shows', 'distance_to_clinic_km', 'prior_no_show_rate']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.3f}")

Accuracy:  0.631
Precision: 0.624
Recall:    0.660
F1-score:  0.642
ROC-AUC:   0.677


## Part 1: Week 7 Testing Readiness

**A. Component to be tested:** The Week 6 candidate model (refined 
Logistic Regression, using `booking_lead_category_v2`).

**Intended purpose:** Predict patient no-shows to support proactive 
clinic intervention.

**Requirement it should satisfy:** Adequately use available behavioural 
signals (booking timing, reminder status, patient history) to 
distinguish likely no-shows from likely attendees.

**Connected tracks:** Data Analytics (validated finding to be tested 
below), ML Engineering (candidate model interface, to be communicated 
once this testing cycle concludes).

**B. Testing objective:** Determine whether combining booking lead 
time and reminder status (rather than treating them as two independent 
features) improves the model's ability to predict no-shows, based on a 
validated finding from the Data Analytics track showing the two 
variables interact.

**C. Expected results table:** See test log below.

**D. Evidence:** Independent recalculation of the collaborator's 
finding, an engineered interaction feature, a retrained model, and a 
before/after performance comparison.

## Cross-Track Finding: Booking Lead Time × Reminder Status

The Data Analytics track collaborator shared a combined breakdown 
testing whether the booking lead time pattern holds when reminder 
status is also considered:

| Lead Time | No Reminder | Reminder Sent |
|---|---|---|
| 0–7 days | 31.25% | 28.83% |
| 8–14 days | 37.33% | 34.43% |
| 15–30 days | 49.72% | 43.91% |
| 31–60 days | 67.63% | 62.58% |

Her finding: the lead-time pattern holds regardless of reminder 
status, but within every lead-time group, sending a reminder is 
associated with a lower no-show rate. This is independently verified 
below before being used to inform any modelling decision.

**Why this is the right finding to test now:** Task 10 already showed 
that `reminder_sent` carries real, if modest, signal even within the 
short-lead-time group, and Tasks 6-9 showed the model's recall 
collapses specifically in that same group. Her finding gives a 
concrete, validated way to test whether combining lead time and 
reminder status, rather than leaving them as two separate features, 
helps the model actually use that signal.

In [4]:
# Independently verify her combined finding against this notebook's data
verify = df_model.groupby(['booking_lead_category_v2', 'reminder_sent'])['target_no_show'].mean() * 100
print(verify)

booking_lead_category_v2  reminder_sent
A_0-7                     No               31.250000
                          Yes              28.828829
B_8-14                    No               37.333333
                          Yes              34.433962
C_15-30                   No               49.717514
                          Yes              43.907794
D_31plus                  No               67.632850
                          Yes              62.582188
Name: target_no_show, dtype: float64


## Task 1: Review of Week 6 Candidate Model

**Model:** Logistic Regression, using the refined `booking_lead_category_v2` 
feature (4 bins, validated against the Data Analytics track's finding).

**Week 6 performance (reproduced above):**
- Accuracy: 0.631
- Precision: 0.624
- Recall: 0.660
- F1-score: 0.642
- ROC-AUC: 0.677

**Why this model was selected in Week 6:** It matched the Week 5 
baseline's recall exactly, the priority metric for HealthConnect's use 
case, while using a cross-track-validated feature and remaining fully 
interpretable. A Random Forest alternative was tested and rejected 
despite marginally better accuracy, since it traded away recall.

**Known weakness identified in Week 6 error analysis:** The model's 
false positives (patients wrongly flagged as no-shows) clustered 
around long booking lead time and prior appointment history, while 
false negatives (missed no-shows) clustered around short lead time and 
low prior no-show rate. This suggested the model may be over-relying 
on lead time and history as blunt signals, without enough nuance to 
distinguish genuinely different risk levels within those groups.

## Task 2: Review of Week 6 Testing Requirements

The Week 6 notebook identified the following as required for Week 7:

1. Patient-aware cross-validation (rather than a single train/test 
   split) to obtain more robust performance estimates.
2. Hyperparameter tuning for Random Forest, to determine whether its 
   full potential exceeds the refined Logistic Regression.
3. Feature importance review, to confirm or challenge which variables 
   are most predictive.
4. Classification threshold testing, to evaluate whether recall can be 
   improved without an unacceptable precision trade-off.
5. Validation of the candidate model's input/output interface with the 
   ML Engineering track.

These requirements, together with the new Data Analytics finding on 
booking lead time and reminder status interaction, define the testing 
scope for this notebook.

## Task 3: Testing Approach

This week's testing covers four areas:

1. **Metric-based testing:** Re-confirm the candidate model's 
   performance on the held-out test set (already reproduced above), 
   then examine it more closely through error and segment analysis.
2. **Error and segment analysis:** Investigate false positives/negatives 
   in more depth than Week 6, including whether performance holds 
   consistently across key segments (e.g. booking lead time groups, 
   reminder status).
3. **Overfitting check:** Compare training-set performance against 
   test-set performance to check whether the model generalises 
   appropriately.
4. **Cross-track validation test:** Independently verify the Data 
   Analytics track's combined booking-lead-time-and-reminder-status 
   finding, then test whether encoding that interaction as a new 
   feature meaningfully improves the model, re-testing after the 
   change.

Each test below follows the format: Test/Scenario → Expected Result → 
Actual Result → Pass/Fail → Issue → Action → Retest Result.

## Tasks 4 & 5: Metric-Based Testing

| Test | Expected Result | Actual Result | Pass/Fail |
|---|---|---|---|
| Reproduce Week 6 performance on test set | Accuracy ≈0.631, Recall ≈0.660, ROC-AUC ≈0.677 | Accuracy 0.631, Recall 0.660, ROC-AUC 0.677 | ✅ Pass |

The candidate model reproduces its Week 6 performance exactly, 
confirming the pipeline is stable and consistent going into this 
week's deeper testing.

In [5]:
# Build labelled results table
results = X_test.copy()
results['actual'] = y_test.values
results['predicted'] = y_pred
results['predicted_proba'] = y_pred_proba

false_negatives = results[(results['actual'] == 1) & (results['predicted'] == 0)]
false_positives = results[(results['actual'] == 0) & (results['predicted'] == 1)]

print("False Negatives (missed no-shows):", len(false_negatives))
print("False Positives (wrongly flagged):", len(false_positives))

False Negatives (missed no-shows): 164
False Positives (wrongly flagged): 192


In [6]:
print("=== Profile of False Negatives ===")
print(false_negatives[['booking_lead_days', 'prior_no_show_rate', 'has_prior_history']].describe())

print("\n=== Profile of False Positives ===")
print(false_positives[['booking_lead_days', 'prior_no_show_rate', 'has_prior_history']].describe())

=== Profile of False Negatives ===
       booking_lead_days  prior_no_show_rate  has_prior_history
count         164.000000          164.000000         164.000000
mean           19.140244            0.116885           0.920732
std            11.374758            0.215332           0.270984
min             0.000000            0.000000           0.000000
25%            10.000000            0.000000           1.000000
50%            18.000000            0.000000           1.000000
75%            26.000000            0.200000           1.000000
max            54.000000            1.000000           1.000000

=== Profile of False Positives ===
       booking_lead_days  prior_no_show_rate  has_prior_history
count         192.000000          192.000000         192.000000
mean           38.781250            0.204216           0.984375
std            12.929291            0.262093           0.124344
min             3.000000            0.000000           0.000000
25%            30.000000         

In [7]:
# Assess model performance across booking lead time segments specifically,
# since that's the feature most central to this week's testing
results_full = X_test.copy()
results_full['actual'] = y_test.values
results_full['predicted'] = y_pred

# Reconstruct the lead time category for segment analysis
def get_category(row):
    if row.get('booking_lead_category_v2_B_8-14', 0) == 1:
        return 'B_8-14'
    elif row.get('booking_lead_category_v2_C_15-30', 0) == 1:
        return 'C_15-30'
    elif row.get('booking_lead_category_v2_D_31plus', 0) == 1:
        return 'D_31plus'
    else:
        return 'A_0-7'

results_full['lead_category'] = results_full.apply(get_category, axis=1)

segment_accuracy = results_full.groupby('lead_category').apply(
    lambda x: accuracy_score(x['actual'], x['predicted'])
)
print("Accuracy by booking lead time segment:")
print(segment_accuracy)

segment_recall = results_full.groupby('lead_category').apply(
    lambda x: recall_score(x['actual'], x['predicted']) if x['actual'].sum() > 0 else None
)
print("\nRecall by booking lead time segment:")
print(segment_recall)

Accuracy by booking lead time segment:
lead_category
A_0-7       0.725806
B_8-14      0.716981
C_15-30     0.556777
D_31plus    0.630670
dtype: float64

Recall by booking lead time segment:
lead_category
A_0-7       0.060606
B_8-14      0.161290
C_15-30     0.336134
D_31plus    0.906667
dtype: float64


### Error Analysis Interpretation (Tasks 6-9)

**False Negatives (164):** Short average lead time (19.1 days), low 
prior no-show rate (0.117), but 92% have some prior history. These 
patients look low-risk on the surface, short notice, decent track 
record, but no-show anyway.

**False Positives (192):** Long average lead time (38.8 days), higher 
prior no-show rate (0.204), and 98% have prior history. The model 
over-flags patients who match the "long lead time + has history" 
profile, even when they actually attend.

**Segment-level performance (the key finding):** Accuracy alone hides 
a serious imbalance in how the model behaves across booking lead time 
groups:

| Segment | Accuracy | Recall |
|---|---|---|
| A_0-7 (0-7 days) | 0.726 | 0.061 |
| B_8-14 (8-14 days) | 0.717 | 0.161 |
| C_15-30 (15-30 days) | 0.557 | 0.336 |
| D_31plus (31+ days) | 0.631 | 0.907 |

The model's recall is almost entirely concentrated in the 31+ day 
group (90.7%) and collapses for short-lead-time appointments (6.1% for 
0-7 days, 16.1% for 8-14 days). In practice, this means the model has 
essentially learned "long lead time = no-show" and is barely 
distinguishing attendance risk at all for appointments booked close to 
the appointment date, even though those groups still contain real 
no-shows (confirmed by the earlier no-show rate breakdown: 29.5% for 
0-7 days is still nearly 1 in 3 appointments).

This is a more serious weakness than the overall metrics suggested. A 
model with 66% overall recall sounds reasonably balanced, but that 
number is propped up almost entirely by one segment. For 
short-lead-time appointments specifically, the model is providing 
almost no early-warning value at all, which directly undermines its 
usefulness for HealthConnect's stated goal, since short-notice 
no-shows are arguably the hardest for the clinic to backfill.

## Task 10: Feature Adequacy Review

The segment analysis in Tasks 6-9 shows the model's predictive power 
is heavily concentrated in long-lead-time appointments and nearly 
absent for short-lead-time ones. This raises a specific question: are 
the current features simply insufficient to distinguish risk within 
the short-lead-time group, or is the model failing to use the 
information it already has?

To investigate, it's worth checking whether other features 
(`prior_no_show_rate`, `reminder_sent`) show any meaningful variation 
*within* the short-lead-time group specifically, since if they do, the 
model should in principle be able to use them, but currently isn't 
doing so effectively.

In [8]:
# Check whether other features vary meaningfully within the short-lead-time group
short_lead = df_model[df_model['booking_lead_category_v2'] == 'A_0-7']

print("Within 0-7 day lead time group:")
print("\nNo-show rate by reminder status:")
print(short_lead.groupby('reminder_sent')['target_no_show'].mean() * 100)

print("\nNo-show rate by has_prior_history:")
print(short_lead.groupby('has_prior_history')['target_no_show'].mean() * 100)

print("\nCorrelation of prior_no_show_rate with target, within this group:")
print(short_lead[['prior_no_show_rate', 'target_no_show']].corr().iloc[0,1])

Within 0-7 day lead time group:

No-show rate by reminder status:
reminder_sent
No     31.250000
Yes    28.828829
Name: target_no_show, dtype: float64

No-show rate by has_prior_history:
has_prior_history
0    20.000000
1    29.879102
Name: target_no_show, dtype: float64

Correlation of prior_no_show_rate with target, within this group:
0.1195549723170636


### Task 10 Conclusion: Features Are Adequate, the Model Isn't Using 
Them Well

Within the 0-7 day lead time group specifically, both remaining 
features still show real variation in no-show rate:

- **Reminder status:** 31.3% (no reminder) vs 28.8% (reminder sent), a 
  smaller but still present gap, consistent with the overall pattern.
- **Prior history:** 20.0% (no history) vs 29.9% (has history), a 
  meaningful 10-point difference.
- **Prior no-show rate:** a positive correlation (0.12) with the 
  target within this group, weak but non-zero, confirming some 
  genuine signal exists.

**This is an important distinction.** The features are not 
uninformative for short-lead-time appointments, they still separate 
higher-risk from lower-risk patients to a meaningful degree. The 
model's near-zero recall in this segment (6.1%) is therefore better 
explained as a **modelling weakness**, not a data limitation: with 
booking lead time dominating the overall feature space so strongly, 
the model's learned decision boundary appears to be under-weighting 
the more subtle within-group signals from reminder status and prior 
history when lead time itself is short.

This directly motivates two things going into refinement: 
(1) testing an interaction feature between booking lead time and 
reminder status, which is exactly the cross-track finding the Data 
Analytics track has independently validated, and (2) considering 
whether the current feature scaling or model weighting is allowing 
lead time to dominate at the expense of these other signals.

In [9]:
y_train_pred = model.predict(X_train_scaled)

print("=== Training Set Performance ===")
print(f"Accuracy:  {accuracy_score(y_train, y_train_pred):.3f}")
print(f"Precision: {precision_score(y_train, y_train_pred):.3f}")
print(f"Recall:    {recall_score(y_train, y_train_pred):.3f}")
print(f"F1-score:  {f1_score(y_train, y_train_pred):.3f}")

print("\n=== Test Set Performance (for comparison) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")

=== Training Set Performance ===
Accuracy:  0.635
Precision: 0.640
Recall:    0.660
F1-score:  0.650

=== Test Set Performance (for comparison) ===
Accuracy:  0.631
Precision: 0.624
Recall:    0.660
F1-score:  0.642


### Task 11 Conclusion: No Evidence of Overfitting

Training and test set performance are nearly identical (accuracy 0.635 
vs 0.631, recall 0.660 vs 0.660, F1 0.650 vs 0.642). This is not the 
pattern of an overfit model, which would typically show noticeably 
higher training performance than test performance. Instead, this 
similarity suggests the model may actually be **underfitting** the 
short-lead-time segment specifically, it isn't memorising noise in the 
training data, it simply hasn't learned enough distinguishing signal 
for that group, consistent with the Task 10 finding that usable signal 
exists there but isn't being captured.

**Test log entry:**

| Test | Expected | Actual | Pass/Fail |
|---|---|---|---|
| Train vs test performance gap | Small gap (no overfitting) | Accuracy gap 0.004, Recall gap 0.000 | ✅ Pass |

## Tasks 12, 13 & 14: Baseline Comparison and Remaining Weaknesses

**Comparison against Week 5 baseline:**

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---|---|---|---|---|
| Week 5 Baseline (3-bin) | 0.631 | 0.624 | 0.660 | 0.642 | 0.677 |
| Week 6 Candidate (4-bin, refined) | 0.631 | 0.624 | 0.660 | 0.642 | 0.677 |

**Is the improvement meaningful?** At the aggregate level, no, the 
Week 6 refinement produced an identical result to the Week 5 baseline 
on every metric. This was already documented honestly in Week 6.

**What Week 7's segment analysis adds:** The aggregate comparison was 
always going to look flat, because it was hiding the real story. The 
segment-level testing this week reveals that the model's apparent 
"reasonable" 66% recall is not evenly earned, it comes almost entirely 
from the 31+ day group (90.7% recall) while the 0-7 day group sits at 
just 6.1% recall. This is a more useful and more concerning finding 
than Week 6's flat aggregate comparison suggested, and it would not 
have been visible without this week's deeper segment testing.

**Remaining weaknesses identified:**
1. Severely imbalanced recall across booking lead time segments, 
   concentrated almost entirely in the longest-lead-time group.
2. The model appears to under-weight reminder status and prior history 
   signals specifically when lead time is short, despite those signals 
   carrying real information.
3. No hyperparameter tuning or threshold adjustment has yet been 
   attempted to address this imbalance.

## Task 15: Incorporating the Data Analytics Finding

### Cross-Track Validation Confirmed

The combined no-show rates calculated independently in this notebook 
match the Data Analytics track's shared finding exactly (31.25%, 
28.83%, 37.33%, 34.43%, 49.72%, 43.91%, 67.63%, 62.58%). Both tracks 
are confirmed to be working from a consistent, correctly-processed 
dataset, and the finding is validated for use in feature refinement.

## Task 16: Communication to ML Engineering

The following model requirements are communicated to the ML 
Engineering track ahead of any pipeline integration:

- **Current candidate model:** Logistic Regression, 31 features 
  (post-encoding), requiring `StandardScaler`-transformed numerical 
  inputs (`age`, `booking_lead_days`, `previous_appointments`, 
  `previous_no_shows`, `distance_to_clinic_km`, `prior_no_show_rate`) 
  alongside one-hot encoded categorical inputs.
- **Known limitation to flag:** The model's recall is highly uneven 
  across booking lead time segments (6.1% for 0-7 day appointments vs 
  90.7% for 31+ day appointments). If a refined version (with an 
  interaction feature) is adopted below, the pipeline's expected input 
  schema will change, ML Engineering should expect a possible feature 
  addition, not just a feature count change.
- **Interface stability:** The target variable encoding (1 = No-Show, 
  0 = Attended) and the patient-aware split logic remain unchanged 
  from Week 6, no disruption expected on that front.

## Tasks 17, 18 & 19: Refinement, Re-Testing, and Before/After Comparison

**Refinement applied:** An interaction feature is engineered combining 
booking lead time category and reminder status, so the model can 
directly learn the combined effect Task 10 and the Data Analytics 
finding both point to, rather than relying on the model to infer the 
interaction from two separate features.

In [10]:
# Engineer the interaction feature
df_model['lead_reminder_interaction'] = (
    df_model['booking_lead_category_v2'].astype(str) + '_' + df_model['reminder_sent'].astype(str)
)

print(df_model['lead_reminder_interaction'].value_counts())

lead_reminder_interaction
D_31plus_Yes    1673
C_15-30_Yes      911
D_31plus_No      621
A_0-7_Yes        444
B_8-14_Yes       424
C_15-30_No       354
A_0-7_No         160
B_8-14_No        150
Name: count, dtype: int64


In [11]:
# Re-encode with the new interaction feature, dropping the original two
# (the interaction replaces them to directly test whether combining helps)
df_model_v3 = df_model.drop(columns=['booking_lead_category_v2', 'reminder_sent'])

categorical_cols_v3 = ['gender', 'age_group', 'appointment_type', 'appointment_day',
                        'appointment_time', 'reminder_channel', 'lead_reminder_interaction']

df_encoded_v3 = pd.get_dummies(df_model_v3, columns=categorical_cols_v3, drop_first=True)

X_v3 = df_encoded_v3.drop(columns=['patient_id', 'target_no_show'])
y_v3 = df_encoded_v3['target_no_show']
groups_v3 = df_encoded_v3['patient_id']

gss3 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx3, test_idx3 = next(gss3.split(X_v3, y_v3, groups_v3))

X_train3, X_test3 = X_v3.iloc[train_idx3], X_v3.iloc[test_idx3]
y_train3, y_test3 = y_v3.iloc[train_idx3], y_v3.iloc[test_idx3]

X_train3_scaled = X_train3.copy()
X_test3_scaled = X_test3.copy()
X_train3_scaled[numerical_cols] = scaler.fit_transform(X_train3[numerical_cols])
X_test3_scaled[numerical_cols] = scaler.transform(X_test3[numerical_cols])

model_refined = LogisticRegression(max_iter=1000, random_state=42)
model_refined.fit(X_train3_scaled, y_train3)

y_pred3 = model_refined.predict(X_test3_scaled)
y_proba3 = model_refined.predict_proba(X_test3_scaled)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test3, y_pred3):.3f}")
print(f"Precision: {precision_score(y_test3, y_pred3):.3f}")
print(f"Recall:    {recall_score(y_test3, y_pred3):.3f}")
print(f"F1-score:  {f1_score(y_test3, y_pred3):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test3, y_proba3):.3f}")

Accuracy:  0.633
Precision: 0.625
Recall:    0.660
F1-score:  0.642
ROC-AUC:   0.677


In [12]:
# Re-check segment-level recall with the refined model
results_v3 = X_test3.copy()
results_v3['actual'] = y_test3.values
results_v3['predicted'] = y_pred3

interaction_cols = [c for c in results_v3.columns if c.startswith('lead_reminder_interaction_')]

def get_lead_from_interaction(row):
    for col in interaction_cols:
        if row[col] == 1:
            if 'A_0-7' in col:
                return 'A_0-7'
            elif 'B_8-14' in col:
                return 'B_8-14'
            elif 'C_15-30' in col:
                return 'C_15-30'
            elif 'D_31plus' in col:
                return 'D_31plus'
    return 'A_0-7'  # reference category (dropped by drop_first)

results_v3['lead_category'] = results_v3.apply(get_lead_from_interaction, axis=1)

segment_recall_v3 = results_v3.groupby('lead_category').apply(
    lambda x: recall_score(x['actual'], x['predicted']) if x['actual'].sum() > 0 else None
)
print("Recall by booking lead time segment (REFINED model):")
print(segment_recall_v3)

Recall by booking lead time segment (REFINED model):
lead_category
A_0-7       0.060606
B_8-14      0.161290
C_15-30     0.327731
D_31plus    0.910000
dtype: float64


### Task 19: Before/After Comparison and Interpretation

| Metric | Original (2 separate features) | Refined (interaction feature) | Change |
|---|---|---|---|
| Accuracy | 0.631 | 0.633 | +0.002 |
| Precision | 0.624 | 0.625 | +0.001 |
| Recall | 0.660 | 0.660 | 0.000 |
| F1-score | 0.642 | 0.642 | 0.000 |
| ROC-AUC | 0.677 | 0.677 | 0.000 |

**Segment-level recall, before vs after refinement:**

| Segment | Before | After | Change |
|---|---|---|---|
| A_0-7 | 0.061 | 0.061 | 0.000 |
| B_8-14 | 0.161 | 0.161 | 0.000 |
| C_15-30 | 0.336 | 0.328 | -0.008 |
| D_31plus | 0.907 | 0.910 | +0.003 |

**Test result: Fail (as a fix for the identified weakness).**

Combining booking lead time and reminder status into a single 
interaction feature did not resolve the short-lead-time recall 
problem. Performance in the 0-7 and 8-14 day segments is essentially 
identical before and after, and the 15-30 day segment even shows a 
small decline. This is a genuine negative result, and it is reported 
as such rather than reframed.

**Why this matters more than it might first appear:** This test 
successfully **rules out** a plausible hypothesis (that the model 
simply needed the two features combined to use them properly) using 
evidence rather than assumption. The real issue is not that lead time 
and reminder status weren't being related to each other in the 
feature set, Logistic Regression can already learn additive effects 
between separate features without needing an explicit interaction 
term. The more likely explanation is that Logistic Regression's linear 
decision boundary is structurally dominated by `booking_lead_days` 
(the strongest single predictor, per Week 5/6 findings), and 
reweighting the *encoding* of two related categorical features doesn't 
change that underlying dynamic.

**Action taken as a result of this negative finding:** No feature 
change is adopted from this test. The original two-feature 
representation (`booking_lead_category_v2` and `reminder_sent` as 
separate features) is retained, since the interaction version offers 
no advantage and adds unnecessary complexity. This is documented as a 
validated negative result for the record, and the segment-level 
recall imbalance is carried forward as an open, unresolved weakness 
into Task 20 and the Week 8 recommendations, likely requiring a 
non-linear model (e.g. Random Forest, revisited with proper tuning) 
or a decision-threshold adjustment specific to the short-lead-time 
segment, rather than further feature engineering.

## Task 20: Assessment of Suitability for the HealthConnect Use Case

The candidate model's aggregate metrics (66% recall, 0.677 ROC-AUC) 
suggested reasonable performance in Week 6. This week's segment-level 
testing shows that assessment was incomplete: the model is highly 
effective for long-lead-time appointments (91% recall) but provides 
almost no early-warning value for short-lead-time appointments (6-16% 
recall).

**Is this suitable for HealthConnect's intended use?** Partially. If 
HealthConnect's primary interest is flagging appointments booked well 
in advance, this model is genuinely useful. But if the clinic also 
wants to catch near-term no-shows (arguably the harder and more 
operationally costly case, since there is less time to fill a 
cancelled slot), this model does not yet deliver that. Deploying it 
without this caveat clearly communicated would risk giving 
HealthConnect false confidence in its coverage of short-notice risk.

**Recommendation:** The model is suitable for a segmented use case 
(flagging long-lead-time appointments) but not yet suitable as a 
general-purpose no-show predictor across all booking timeframes.

## Task 21: Remaining Model Limitations and Risks

- **Segment imbalance (primary risk):** Recall is concentrated almost 
  entirely in the 31+ day lead time group; short-lead-time recall 
  remains largely unaddressed after this week's testing.
- **Interaction feature tested and ruled out:** Combining lead time 
  and reminder status did not resolve the imbalance, narrowing but not 
  yet solving the problem.
- **Linear model ceiling:** The lack of movement from the interaction 
  feature suggests Logistic Regression's linear structure may itself 
  be the limiting factor, not the features available to it.
- **No threshold tuning attempted:** The default 0.5 classification 
  threshold has not been tested against the short-lead-time segment 
  specifically, a segment-specific threshold could plausibly help 
  without requiring a model change.
- **No hyperparameter tuning on Random Forest:** carried over 
  unresolved from Week 6.
- **Single train/test split:** performance estimates still rely on one 
  patient-aware split rather than cross-validation.
- **Synthetic data:** real-world generalisation remains unverified.

## Task 22: What Must Be Completed Before Week 8

1. Test whether a non-linear model (Random Forest, properly tuned this 
   time) closes the short-lead-time recall gap that Logistic 
   Regression's linear boundary cannot.
2. Test a lower classification threshold specifically for the 
   short-lead-time segment, to check if recall can be improved there 
   without unacceptable precision loss.
3. Perform patient-aware cross-validation to confirm these segment-level 
   findings are stable and not an artifact of this particular 
   train/test split.
4. Finalise and communicate the confirmed model interface to ML 
   Engineering once the above is resolved.

## Assumptions, Limitations, Risks and Dependencies (Reviewed Across Weeks 4-7)

**Resolved since earlier weeks:**
- Week 6's uncertainty about whether the refined `booking_lead_category_v2` 
  feature was "good enough" is now better understood: it's necessary 
  but not sufficient, the real bottleneck is the model's linear 
  structure, not the feature encoding.
- Overfitting, flagged as an open question in Week 6, was tested this 
  week and ruled out (train/test performance nearly identical).

**Unresolved:**
- The short-lead-time recall gap remains unresolved after this week's 
  most direct attempt to fix it.
- Real-world generalisation of synthetic data remains unverified.

**New issues discovered in Week 7:**
- Aggregate metrics were actively misleading about where the model's 
  actual strengths and weaknesses lie; segment-level testing was 
  necessary to reveal this, a methodological lesson worth carrying 
  into all future evaluation.
- A validated cross-track hypothesis (combining lead time and reminder 
  status) was tested rigorously and failed to improve the target 
  segment, a legitimate negative result rather than a wasted effort.

**New cross-track dependencies:**
- ML Engineering now has a documented limitation (segment imbalance) 
  to be aware of before finalising pipeline integration.
- The Data Analytics collaborator is owed a direct reply on this 
  test's outcome, since her finding was the basis for it.

**Technical/modelling limitations:** See Task 21.

**Integration/testing challenges:** None encountered technically; the 
main challenge was resisting the temptation to treat a small aggregate 
uptick (accuracy +0.002) as a meaningful win when the segment-level 
data showed otherwise.

**Potential impact on the project:** If left unresolved, HealthConnect 
would receive a model that appears to perform reasonably overall but 
systematically underperforms on short-notice appointments, likely the 
category where proactive intervention is most valuable.

**Recommended mitigation:** Prioritise non-linear model testing and 
threshold tuning in Week 8, as outlined in Task 22, before any 
recommendation to deploy.